# **Preprocesamiento - Procesamiento de Lenguaje Natural**

## **1. Librerias**

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Descargas NLTK
nltk.download('punkt')
nltk.download('stopwords')

I0000 00:00:1778470558.784931   62317 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778470559.739240   62317 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/guirlessa/tf_gpu/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core p

True

## **2. Cargar Dataset**

In [2]:
df = pd.read_csv('../Data/Resume/Resume_eda.csv')
df.head()

,ID,Resume_str,Resume_html,Category,Length,Word_count,mean_word_length,mean_sent_length
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,5442,674,6.459941,205.538462
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,5572,708,6.307910,202.074074
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,7720,1017,6.218289,199.921053
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,2855,379,5.997361,164.823529
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,9172,1206,6.368159,164.236364


In [3]:
df['clean_text'] = df['Resume_str'].astype(str)

df[['Resume_str', 'clean_text']].head()

,Resume_str,clean_text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ...","HR SPECIALIST, US HR OPERATIONS ..."
2,HR DIRECTOR Summary Over 2...,HR DIRECTOR Summary Over 2...
3,HR SPECIALIST Summary Dedica...,HR SPECIALIST Summary Dedica...
4,HR MANAGER Skill Highlights ...,HR MANAGER Skill Highlights ...


## **3. Preprocesamiento**

### **3.1. Minúsculas**

In [4]:
df['clean_text'] = df['clean_text'].str.lower()

df['clean_text'].head()

0             hr administrator/marketing associate\...
1             hr specialist, us hr operations      ...
2             hr director       summary      over 2...
3             hr specialist       summary    dedica...
4             hr manager         skill highlights  ...
Name: clean_text, dtype: object

### **3.2. Eliminando Símbolos**

In [5]:
def remove_symbols(text):

    # eliminar caracteres especiales y números
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df['clean_text'] = df['clean_text'].apply(remove_symbols)

df['clean_text'].head()

0    hr administrator marketing associate hr admini...
1    hr specialist us hr operations summary versati...
2    hr director summary over years experience in r...
3    hr specialist summary dedicated driven and dyn...
4    hr manager skill highlights hr skills hr depar...
Name: clean_text, dtype: object

### **3.3. Tokenización**

In [6]:
df['tokens'] = df['clean_text'].apply(word_tokenize)

df['tokens'].head()

0    [hr, administrator, marketing, associate, hr, ...
1    [hr, specialist, us, hr, operations, summary, ...
2    [hr, director, summary, over, years, experienc...
3    [hr, specialist, summary, dedicated, driven, a...
4    [hr, manager, skill, highlights, hr, skills, h...
Name: tokens, dtype: object

#### **3.3.1. Eliminando StopWordds**

In [7]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):

    filtered = [
        word for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return filtered

df['tokens'] = df['tokens'].apply(remove_stopwords)

df['tokens'].head()

0    [administrator, marketing, associate, administ...
1    [specialist, operations, summary, versatile, m...
2    [director, summary, years, experience, recruit...
3    [specialist, summary, dedicated, driven, dynam...
4    [manager, skill, highlights, skills, departmen...
Name: tokens, dtype: object

#### **3.3.2. Convertir Tokens a Texto**

In [8]:

df['processed_text'] = df['tokens'].apply(
    lambda tokens: ' '.join(tokens)
)

df[['processed_text']].head()

,processed_text
0,administrator marketing associate administrato...
1,specialist operations summary versatile media ...
2,director summary years experience recruiting p...
3,specialist summary dedicated driven dynamic ye...
4,manager skill highlights skills department sta...


In [9]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts(df['processed_text'])

# Convertir texto a secuencias numéricas
sequences = tokenizer.texts_to_sequences(
    df['processed_text']
)

print(sequences[0][:20])


[680, 20, 178, 680, 88, 777, 7, 11, 18, 81, 15, 1533, 7, 11, 4, 4234, 2580, 341, 7, 491]


### **3.4. Padding**

In [10]:
# Longitud de cada secuencia
seq_lengths = [len(seq) for seq in sequences]

# Estadísticas
pd.Series(seq_lengths).describe()

count    2484.000000
mean      584.221820
std       260.676766
min         0.000000
25%       476.000000
50%       547.000000
75%       674.000000
max      3543.000000
dtype: float64

El tamaño máximo de padding fue definido a partir de la distribución de longitudes de las secuencias tokenizadas. Dado que el 75% de los resumes contiene menos de 674 tokens y la media es de aproximadamente 584 tokens, se seleccionó un tamaño de 700 tokens para conservar la mayor parte de la información textual, manteniendo un equilibrio entre capacidad representativa y costo computacional.

In [11]:
max_length = 700

X_padded = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X_padded.shape)

print(X_padded[0])

(2484, 700)
[  680    20   178   680    88   777     7    11    18    81    15  1533
     7    11     4  4234  2580   341     7   491   238  8218 10467  1710
  1409  1905     7    11   220   491     7   266    12     4    20  2033
   906   396   348    14    13   558   272  2060    37    92   191   208
  3308  3660   283    14   427   196 10468     7  1682    20  3438  5694
  2474   106    18    14   427  1329   835   405   247  1533    30  5694
 14355 12630  8219  5990 22354  8219  4526    29 14356 22355   234   773
  1212     7    11     6   173    67    89   861  1278    87    55     4
    63    60   664    15   680    20   178   680  1216    49     2     5
     3     1  4380   127    96  4381  2344    79   646   733   689    92
   276    14    78    46  1776    78  4956  3026   184   159  5298   154
   276   104  1512   230  4527   143   134   385  1107   672  1703    60
   542  2361    78  1246   395   102  7849     4   203   396    78    92
    70  6535   276    83   504    84  1

## **4. Embeddings**

### **4.1. Word2Vec**

In [13]:
from gensim.models import Word2Vec

# Entrenar modelo
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

# Resumen
print(w2v_model)

Word2Vec<vocab=22353, vector_size=100, alpha=0.025>


In [14]:
w2v_model.wv['management']

array([-1.5833719 , -0.20249546,  1.1301259 , -1.4413348 ,  0.91480947,
       -3.079953  , -0.85435367,  1.5407827 , -0.11971991, -0.20905346,
        0.0455852 ,  0.4721243 ,  1.1730391 , -0.5621716 ,  0.42228803,
       -2.5007315 ,  0.52342   , -0.44808948, -1.5347165 ,  0.83665156,
       -0.44337407,  1.6048868 , -0.60487   , -0.23617099, -0.47106406,
        0.5959619 ,  1.8293943 , -0.9421234 ,  1.3510243 , -0.58603185,
       -1.1979569 ,  0.43486694,  0.00968385,  0.2666047 ,  0.56072986,
        2.919185  ,  1.2382282 , -1.1325506 , -1.1650496 ,  0.29201666,
        2.306105  ,  0.3472352 , -0.33657843, -0.66798127,  1.5892508 ,
       -2.3501952 , -1.8607718 , -0.7738825 , -0.80923986, -1.245237  ,
        2.0779598 , -0.35504878, -0.11660109, -1.5862734 , -0.67912364,
        0.04087362, -0.479918  , -0.29150495,  1.7552049 ,  1.8888602 ,
       -1.2979541 , -0.6178631 ,  1.0217952 , -0.27117833,  3.682534  ,
       -0.9242898 ,  0.09709413,  1.486116  , -0.7628399 , -0.10

In [15]:
w2v_model.wv.most_similar('management')

[('leadership', 0.6235785484313965),
 ('budgeting', 0.6043535470962524),
 ('negotiations', 0.5827123522758484),
 ('negotiation', 0.5804188847541809),
 ('coordination', 0.5755197405815125),
 ('planning', 0.5666571259498596),
 ('administration', 0.5577354431152344),
 ('oversight', 0.537636399269104),
 ('forecasting', 0.5307595729827881),
 ('expertise', 0.516238272190094)]

### **4.2. FastText**

In [16]:
from gensim.models import FastText

# Entrenar modelo
fasttext_model = FastText(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print(fasttext_model)

FastText<vocab=22353, vector_size=100, alpha=0.025>


In [17]:
fasttext_model.wv['management']

array([-0.47189036, -2.733503  , -0.9675329 , -1.1452539 , -1.5301622 ,
        0.4785875 , -0.19880556,  0.16635041, -1.5851439 ,  0.770972  ,
       -1.6948155 , -1.4845337 , -0.03734889, -0.36280766,  1.006711  ,
        0.72771394,  0.02393697,  0.7438543 ,  1.4006636 ,  0.8528426 ,
       -1.2168607 ,  2.2857962 ,  1.1009542 ,  0.78964996,  0.18730998,
        1.5346255 ,  1.9626242 , -1.7615827 , -2.0565386 ,  1.9556113 ,
       -1.2121304 ,  0.7286704 , -0.5885561 ,  0.52637607,  1.96163   ,
       -1.5253503 ,  3.375417  , -1.2358161 ,  2.9730427 , -0.14277902,
       -1.7251956 ,  0.6506766 , -1.2971765 ,  0.4739089 ,  1.0239314 ,
        0.7752241 ,  0.38767612,  0.20037182,  0.8510089 , -0.13998647,
        0.9463856 ,  0.5020047 , -2.2192307 ,  0.9485838 ,  1.7754561 ,
        0.05234502,  0.52622867,  0.9071998 ,  0.60563475,  1.0692627 ,
        1.9556283 , -1.369237  ,  1.0150923 ,  0.33525133,  0.7846665 ,
       -2.062654  , -0.0526616 , -1.5176771 , -2.2582715 , -0.07

In [18]:
fasttext_model.wv.most_similar('management')

[('managements', 0.9139692187309265),
 ('managment', 0.9119823575019836),
 ('managament', 0.883163571357727),
 ('mangement', 0.8589568138122559),
 ('managementcoursework', 0.8564959764480591),
 ('judgement', 0.8296743631362915),
 ('managerial', 0.7863560318946838),
 ('infringement', 0.7562222480773926),
 ('imanage', 0.7512562274932861),
 ('engagement', 0.7407241463661194)]